In [1]:
import pandas as pd
import numpy as np


forecast = pd.read_csv('../outputs/forecast_results.csv', 
                        parse_dates=['date'])
risk = pd.read_csv('../outputs/risk_summary.csv')

print("=== Forecast Data ===")
print(forecast.head())
print(f"\n=== Risk Summary ===")
print(risk.to_string())

=== Forecast Data ===
        date  demand  predicted_demand
0 2023-01-01   135.0            126.14
1 2023-01-02   124.0            130.36
2 2023-01-03   138.0            130.12
3 2023-01-04   118.0            129.19
4 2023-01-05   145.0            128.58

=== Risk Summary ===
   mean_demand  std_demand        cv   risk_level  risk_adjusted_demand  forecast_bias  var_95  mean_forecast_error
0   142.953425   14.701257  0.102839  Medium Risk            142.953425      12.229945  31.992            14.356795


In [2]:
def get_financing_recommendation(
    production_cost,
    retail_price,
    risk_level,
    risk_adjusted_demand,
    mean_demand
):
    """
    Core recommendation engine based on Chen, Lu & Cai (2020).
    
    Parameters:
    -----------
    production_cost     : cost per unit for manufacturer
    retail_price        : selling price per unit
    risk_level          : 'Low Risk', 'Medium Risk', 'High Risk'
    risk_adjusted_demand: demand estimate adjusted for uncertainty
    mean_demand         : raw mean demand forecast
    
    Returns:
    --------
    recommendation dict with method and justification
    """
    
    
    cost_ratio = production_cost / retail_price
    
    
    risk_modifier = {
        'Low Risk': 0,       
        'Medium Risk': 0.05, 
        'High Risk': 0.10    
    }[risk_level]
    
  
    low_threshold = 0.40 + risk_modifier
    high_threshold = 0.70 + risk_modifier
    
   
    if cost_ratio <= low_threshold:
        method = 'Early Payment'
        rationale = (
            f"Production cost is low relative to retail price "
            f"(cost ratio: {cost_ratio:.2f}). "
            f"Benefit from increased production outweighs interest income. "
            f"Risk level is {risk_level}, demand forecast: "
            f"{risk_adjusted_demand:.0f} units."
        )
        
    elif cost_ratio <= high_threshold:
        method = 'In-House Factoring'
        rationale = (
            f"Production cost is moderate relative to retail price "
            f"(cost ratio: {cost_ratio:.2f}). "
            f"Retailer can earn interest while maintaining production levels. "
            f"Risk level is {risk_level}, demand forecast: "
            f"{risk_adjusted_demand:.0f} units."
        )
        
    else:
        method = 'Bank Financing'
        rationale = (
            f"Production cost is high relative to retail price "
            f"(cost ratio: {cost_ratio:.2f}). "
            f"Financial risk too high for retailer to absorb. "
            f"Risk level is {risk_level}, demand forecast: "
            f"{risk_adjusted_demand:.0f} units."
        )
    
    return {
        'recommended_method': method,
        'cost_ratio': cost_ratio,
        'risk_level': risk_level,
        'forecast_demand': risk_adjusted_demand,
        'rationale': rationale
    }

print("Recommendation engine loaded successfully")

Recommendation engine loaded successfully


In [3]:

risk_level = risk['risk_level'][0]
risk_adjusted_demand = risk['risk_adjusted_demand'][0]
mean_demand = risk['mean_demand'][0]

retail_price = 20.0


scenarios = [
    {'name': 'Low Cost Supplier',    'cost': 4.0},
    {'name': 'Medium Cost Supplier', 'cost': 10.0},
    {'name': 'High Cost Supplier',   'cost': 16.0},
]

print("=" * 60)
print("FINANCING RECOMMENDATION REPORT")
print("=" * 60)

for scenario in scenarios:
    result = get_financing_recommendation(
        production_cost=scenario['cost'],
        retail_price=retail_price,
        risk_level=risk_level,
        risk_adjusted_demand=risk_adjusted_demand,
        mean_demand=mean_demand
    )
    
    print(f"\nSupplier: {scenario['name']}")
    print(f"Production Cost: ${scenario['cost']}/unit")
    print(f"Retail Price:    ${retail_price}/unit")
    print(f"Cost Ratio:      {result['cost_ratio']:.2f}")
    print(f"Recommendation:  {result['recommended_method']}")
    print(f"Rationale:       {result['rationale']}")
    print("-" * 60)

FINANCING RECOMMENDATION REPORT

Supplier: Low Cost Supplier
Production Cost: $4.0/unit
Retail Price:    $20.0/unit
Cost Ratio:      0.20
Recommendation:  Early Payment
Rationale:       Production cost is low relative to retail price (cost ratio: 0.20). Benefit from increased production outweighs interest income. Risk level is Medium Risk, demand forecast: 143 units.
------------------------------------------------------------

Supplier: Medium Cost Supplier
Production Cost: $10.0/unit
Retail Price:    $20.0/unit
Cost Ratio:      0.50
Recommendation:  In-House Factoring
Rationale:       Production cost is moderate relative to retail price (cost ratio: 0.50). Retailer can earn interest while maintaining production levels. Risk level is Medium Risk, demand forecast: 143 units.
------------------------------------------------------------

Supplier: High Cost Supplier
Production Cost: $16.0/unit
Retail Price:    $20.0/unit
Cost Ratio:      0.80
Recommendation:  Bank Financing
Rationale:   

In [4]:

all_results = []

for scenario in scenarios:
    result = get_financing_recommendation(
        production_cost=scenario['cost'],
        retail_price=retail_price,
        risk_level=risk_level,
        risk_adjusted_demand=risk_adjusted_demand,
        mean_demand=mean_demand
    )
    result['supplier'] = scenario['name']
    result['production_cost'] = scenario['cost']
    result['retail_price'] = retail_price
    all_results.append(result)

recommendations_df = pd.DataFrame(all_results)
recommendations_df.to_csv('../outputs/recommendations.csv', index=False)

print("Recommendations saved successfully!")
print(recommendations_df[['supplier', 'production_cost', 
                           'cost_ratio', 'recommended_method']])

Recommendations saved successfully!
               supplier  production_cost  cost_ratio  recommended_method
0     Low Cost Supplier              4.0         0.2       Early Payment
1  Medium Cost Supplier             10.0         0.5  In-House Factoring
2    High Cost Supplier             16.0         0.8      Bank Financing


In [5]:
import pandas as pd


df = pd.read_csv('../outputs/forecast_results.csv', parse_dates=['date'])


test_upload = df[['date', 'demand']].copy()


test_upload.to_csv('../outputs/test_upload.csv', index=False)

print("Test upload file created!")
print(test_upload.head())
print(f"Total rows: {len(test_upload)}")

Test upload file created!
        date  demand
0 2023-01-01   135.0
1 2023-01-02   124.0
2 2023-01-03   138.0
3 2023-01-04   118.0
4 2023-01-05   145.0
Total rows: 365
